In [1]:
import numpy as np
import pandas as pd
import random
import textwrap

# Pretty display
pd.set_option('display.max_colwidth', 80)
print('Libraries loaded!')

Libraries loaded!


In [2]:
# -------------------------------------------------------
# OPTION A: Type your topics manually
# -------------------------------------------------------
topics = [
    "Compare and contrast the importance of self-reliance and adaptability in healthcare.",
    "Evaluate the effectiveness of management consulting in addressing conflicts within marketing.",
    "Discuss the role of self-reliance in achieving success in software engineering.",
]

# -------------------------------------------------------
# OPTION B: Load from your CSV (uncomment if needed)
# -------------------------------------------------------
# from google.colab import files
# uploaded = files.upload()          # upload test.csv
# df_test = pd.read_csv('test.csv')
# topics = df_test['topic'].tolist()

print(f'Total topics: {len(topics)}')
for i, t in enumerate(topics):
    print(f'  [{i+1}] {t[:80]}...' if len(t) > 80 else f'  [{i+1}] {t}')

Total topics: 3
  [1] Compare and contrast the importance of self-reliance and adaptability in healthc...
  [2] Evaluate the effectiveness of management consulting in addressing conflicts with...
  [3] Discuss the role of self-reliance in achieving success in software engineering.


In [3]:
# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
USE_MODEL   = False      # Set True to use Gemma-2 (requires GPU runtime)
APPROACH    = 'dialectical'   # 'philosophical' | 'metaphorical' | 'dialectical'
NUM_JUDGES  = 5
CRITERIA    = ['Introduction', 'Logical Argument', 'Emotional Argument',
               'Philosophical Depth', 'Conclusion']

print(f'USE_MODEL : {USE_MODEL}')
print(f'APPROACH  : {APPROACH}')
print(f'Judges    : {NUM_JUDGES}')
print(f'Criteria  : {CRITERIA}')

USE_MODEL : False
APPROACH  : dialectical
Judges    : 5
Criteria  : ['Introduction', 'Logical Argument', 'Emotional Argument', 'Philosophical Depth', 'Conclusion']


In [4]:
# -------------------------------------------------------
# ESSAY GENERATION
# -------------------------------------------------------

def build_prompt(topic: str, approach: str) -> str:
    """Build a generation prompt based on approach."""
    instructions = {
        'philosophical': (
            f'Write a philosophical essay (~130 words) on: "{topic}". '
            'Use abstract reasoning, question assumptions, present morally grey arguments, '
            'mix formal language with occasional plain speech, '
            'and end with an ethically ambiguous conclusion. Use perfect grammar.'
        ),
        'metaphorical': (
            f'Write a metaphorical essay (~130 words) on: "{topic}". '
            'Use extended metaphors and rich imagery, draw unexpected connections, '
            'layer multiple levels of meaning, and end with an ethically ambiguous conclusion. '
            'Maintain perfect grammar.'
        ),
        'dialectical': (
            f'Write a dialectical essay (~130 words) on: "{topic}". '
            'Present competing viewpoints, explore tensions between perspectives, '
            'use sophisticated but clear reasoning, '
            'and end with an ethically ambiguous conclusion. Maintain logical structure.'
        ),
    }
    return instructions[approach]


# --- Template fallback (no model needed) ---
TEMPLATES = [
    (
        "The paradox of {topic_lower} lies not in its resolution but in its persistence. "
        "Some argue that rigid principles form the bedrock of progress — without them, "
        "civilizations crumble into relativism. Yet others contend that those same principles "
        "become chains, strangling innovation beneath the weight of tradition. "
        "Is the expert who refuses to adapt a guardian of truth, or merely a fossil? "
        "And is the one who bends to every new wind a pragmatist — or simply spineless? "
        "Perhaps, unsettlingly, both are correct. The tension between them may not be a "
        "problem to be solved but a condition to be endured — the very friction that, "
        "however painfully, keeps systems honest."
    ),
    (
        "Consider {topic_lower} as a river and a dam. The river — wild, generative, "
        "indifferent — represents raw potential. The dam channels it, makes it useful, "
        "but at a cost: the valley upstream drowns. Which matters more, the village that "
        "flourishes downstream or the one submerged? Metrics celebrate the former; "
        "history mourns the latter, briefly. We build efficiency on erasure and call it "
        "progress. The uncomfortable truth is that no framework fully captures this trade-off — "
        "every model optimises for something by neglecting something else. "
        "To act is to choose whose loss is acceptable. That choice, rarely made explicitly, "
        "is the real subject of any serious inquiry into {topic_lower}."
    ),
    (
        "Thesis: {topic_lower} demands clarity, structure, and measurable outcomes. "
        "Antithesis: it demands empathy, ambiguity, and irreducible human judgment. "
        "Synthesis: neither alone is sufficient, yet combining them breeds its own contradictions — "
        "a bureaucracy of feeling, or a sentimentalised algorithm. "
        "Practitioners who lean entirely on data risk missing what the data was never designed to capture; "
        "those who trust instinct alone reproduce their own biases at scale. "
        "The field advances not by resolving this tension but by oscillating between its poles, "
        "each swing correcting the excesses of the last — a dialectic with no final destination, "
        "only the ongoing, imperfect work of thinking better."
    ),
]

def template_essay(topic: str) -> str:
    """Generate a disagreement-optimised essay from a template."""
    tpl = random.choice(TEMPLATES)
    short = topic.rstrip('.').lower()
    return tpl.format(topic_lower=short)


# --- Model-based generation (Gemma-2) ---
model, tokenizer = None, None

def load_model():
    global model, tokenizer
    if model is not None:
        return
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
    model_path = 'google/gemma-2-2b-it'   # change to local path on Kaggle
    print('Loading tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print('Loading model (this may take a minute)...')
    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map='auto', torch_dtype='auto'
    )
    print('Model ready!')

def model_essay(topic: str, approach: str) -> str:
    load_model()
    import torch
    prompt = build_prompt(topic, approach)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()


def generate_essay(topic: str, approach: str = APPROACH) -> str:
    if USE_MODEL:
        return model_essay(topic, approach)
    else:
        return template_essay(topic)


print('Essay generator functions defined.')

Essay generator functions defined.


In [5]:
def simulate_judges(essay: str, n_judges: int = NUM_JUDGES) -> dict:
    """
    Simulate n_judges scoring the essay on each criterion.
    Returns per-criterion avg and std dev (disagreement).
    """
    results = {}
    for criterion in CRITERIA:
        scores = [round(random.uniform(2, 10), 2) for _ in range(n_judges)]
        results[criterion] = {
            'scores' : scores,
            'avg'    : round(np.mean(scores), 3),
            'stdev'  : round(np.std(scores), 3),
        }
    return results


def final_score(judge_data: dict) -> dict:
    """
    Calculate the final disagreement index.
    Higher = more judge disagreement = better for the competition.
    Formula: avg_h * min_v * (9 - avg_q)
    """
    avgs   = [v['avg']   for v in judge_data.values()]
    stdevs = [v['stdev'] for v in judge_data.values()]

    avg_q = round(np.mean(avgs),   3)   # average quality
    avg_h = round(np.mean(stdevs), 3)   # average horizontal disagreement
    min_v = round(np.min(stdevs),  3)   # minimum disagreement (weakest link)
    score = round(avg_h * min_v * (9 - avg_q), 4)

    return {'avg_quality': avg_q, 'avg_disagreement': avg_h,
            'min_disagreement': min_v, 'final_score': score}


print('Scoring functions defined.')

Scoring functions defined.


In [6]:
records = []

for i, topic in enumerate(topics):
    print(f'[{i+1}/{len(topics)}] {topic[:60]}...' if len(topic) > 60 else f'[{i+1}/{len(topics)}] {topic}')

    essay      = generate_essay(topic, APPROACH)
    judge_data = simulate_judges(essay)
    scores     = final_score(judge_data)

    records.append({
        'topic'           : topic,
        'approach'        : APPROACH,
        'essay_preview'   : essay[:120].replace('\n', ' ') + '...',
        'word_count'      : len(essay.split()),
        'avg_quality'     : scores['avg_quality'],
        'avg_disagreement': scores['avg_disagreement'],
        'min_disagreement': scores['min_disagreement'],
        'final_score'     : scores['final_score'],
        '_essay_full'     : essay,
        '_judge_data'     : judge_data,
    })

df = pd.DataFrame(records)
print('\nDone!')

[1/3] Compare and contrast the importance of self-reliance and ada...
[2/3] Evaluate the effectiveness of management consulting in addre...
[3/3] Discuss the role of self-reliance in achieving success in so...

Done!


In [7]:
# Summary table
summary_cols = ['topic', 'approach', 'word_count',
                'avg_quality', 'avg_disagreement', 'min_disagreement', 'final_score']

print('=' * 70)
print('RESULTS SUMMARY')
print('=' * 70)
display(df[summary_cols].style
    .highlight_max(subset=['final_score', 'avg_disagreement'], color='#d4edda')
    .highlight_min(subset=['final_score', 'avg_disagreement'], color='#f8d7da')
    .format({'avg_quality': '{:.3f}', 'avg_disagreement': '{:.3f}',
             'min_disagreement': '{:.3f}', 'final_score': '{:.4f}'})
)

RESULTS SUMMARY


,topic,approach,word_count,avg_quality,avg_disagreement,min_disagreement,final_score
0,Compare and contrast the importance of self-reliance and adaptability in healthcare.,dialectical,112,6.263,1.658,1.042,4.7285
1,Evaluate the effectiveness of management consulting in addressing conflicts within marketing.,dialectical,128,5.455,1.876,1.156,7.6879
2,Discuss the role of self-reliance in achieving success in software engineering.,dialectical,128,5.854,1.727,1.182,6.4220


In [8]:
# Per-criterion breakdown for each essay
print('=' * 70)
print('PER-CRITERION JUDGE SCORES')
print('=' * 70)

for rec in records:
    print(f"\nTopic   : {rec['topic'][:70]}")
    print(f"Approach: {rec['approach']}  |  Words: {rec['word_count']}  |  Final score: {rec['final_score']}")
    print(f"{'Criterion':<25} {'Avg':>6} {'Std':>6}  {'Scores'}")
    print('-' * 65)
    for crit, vals in rec['_judge_data'].items():
        bar = '█' * int(vals['avg'])
        scores_str = '  '.join(str(s) for s in vals['scores'])
        print(f"{crit:<25} {vals['avg']:>6.2f} {vals['stdev']:>6.2f}  [{scores_str}]")
    print()

PER-CRITERION JUDGE SCORES

Topic   : Compare and contrast the importance of self-reliance and adaptability 
Approach: dialectical  |  Words: 112  |  Final score: 4.7285
Criterion                    Avg    Std  Scores
-----------------------------------------------------------------
Introduction                4.78   2.18  [8.87  4.9  3.63  2.53  3.99]
Logical Argument            5.16   1.79  [5.58  6.22  7.68  2.82  3.48]
Emotional Argument          7.21   1.04  [6.69  5.77  8.94  7.13  7.51]
Philosophical Depth         7.83   1.26  [7.53  5.62  8.71  7.97  9.31]
Conclusion                  6.34   2.02  [7.48  2.9  7.67  8.41  5.24]


Topic   : Evaluate the effectiveness of management consulting in addressing conf
Approach: dialectical  |  Words: 128  |  Final score: 7.6879
Criterion                    Avg    Std  Scores
-----------------------------------------------------------------
Introduction                5.30   2.58  [7.6  2.03  8.95  3.48  4.46]
Logical Argument            6

In [9]:
# Print full essays
print('=' * 70)
print('FULL ESSAYS')
print('=' * 70)

for rec in records:
    print(f"\n[Topic] {rec['topic']}")
    print(f"[Approach: {rec['approach']} | Words: {rec['word_count']}]")
    print('-' * 70)
    print(textwrap.fill(rec['_essay_full'], width=80))
    print()

FULL ESSAYS

[Topic] Compare and contrast the importance of self-reliance and adaptability in healthcare.
[Approach: dialectical | Words: 112]
----------------------------------------------------------------------
Thesis: compare and contrast the importance of self-reliance and adaptability in
healthcare demands clarity, structure, and measurable outcomes. Antithesis: it
demands empathy, ambiguity, and irreducible human judgment. Synthesis: neither
alone is sufficient, yet combining them breeds its own contradictions — a
bureaucracy of feeling, or a sentimentalised algorithm. Practitioners who lean
entirely on data risk missing what the data was never designed to capture; those
who trust instinct alone reproduce their own biases at scale. The field advances
not by resolving this tension but by oscillating between its poles, each swing
correcting the excesses of the last — a dialectic with no final destination,
only the ongoing, imperfect work of thinking better.


[Topic] Evaluate the 

In [10]:
# Build submission in the same format as the competition
submission = pd.DataFrame({
    'id'   : range(1, len(records) + 1),
    'essay': [r['_essay_full'] for r in records]
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
display(submission.head())

Saved submission.csv


,id,essay
0,1,Thesis: compare and contrast the importance of self-reliance and adaptabilit...
1,2,Consider evaluate the effectiveness of management consulting in addressing c...
2,3,Consider discuss the role of self-reliance in achieving success in software ...
